# 1. Core Concepts

Physical equality is bitwise identity: the same names, types, and values. Logical equality is the relation you declare — after a rename, a currency strip, and a sentinel, two exports of the same accounts can still be the same data.

This notebook is the Python API: construct a comparison, read `DiffResult`, and isolate a real discrepancy. YAML and CLI are [2. YAML and CLI](02_yaml_and_cli.ipynb). Warehouse routing is in the [Configuration Guide](../configuration.md).

## 1. Two frames that are the same accounts

A legacy export and a warehouse table. Currency symbols, a sentinel for missing status, and a two-cent rounding difference are the only things standing between them.

In [ ]:
import polars as pl

from veridelta import DiffConfig, DiffEngine, DiffRule

source = pl.DataFrame(
    {
        "legacy_id": [1, 2, 3],
        "status": ["Active", "N/A", "Closed"],
        "balance": ["$10.00", "$20.50", "$5.00"],
        "tier": ["Premium", "Standard", "Enterprise"],
    }
)
target = pl.DataFrame(
    {
        "user_id": [1, 2, 3],
        "status": ["Active", None, "Closed"],
        "balance": [10.00, 20.48, 5.00],
        "tier": ["Premium", "Standard", "Enterprise"],
    }
)
print(f"Physical equality: {source.equals(target)}")

# Output:
# Physical equality: False

## 2. Declare the comparison

Rules are not applied in the order you write them. Every column follows the same nine-stage pipeline: sentinels, regex, whitespace and case, value map, padding, datetime, cast, then comparison. `cast_to` therefore always sees already-cleaned text.

In [ ]:
config = DiffConfig(
    primary_keys=["user_id"],
    rules=[
        DiffRule(column_names=["legacy_id"], rename_to="user_id"),
        DiffRule(
            column_names=["balance"],
            regex_replace={"\\$": ""},
            cast_to="Float64",
            absolute_tolerance=0.05,
        ),
        DiffRule(column_names=["status"], null_values=["N/A"], treat_null_as_equal=True),
    ],
)
result = DiffEngine(config, source.lazy(), target.lazy()).run()
print(result.summary.report_summary)

# Output:
# Veridelta Execution Summary
# ===========================
# Status:        PASSED (Perfect Match)
# Match Rate:    100.0%
# Source Rows:   3
# Target Rows:   3
# Volume Shift:  +0 rows
#
# Row-Level Discrepancies:
# ---------------------------
# Added:         0
# Removed:       0
# Changed:       0
# Total Issues:  0

## 3. Read `DiffResult`

`run()` returns a `DiffResult`. `.summary` is the JSON-serializable verdict. `.added`, `.removed`, and `.changed` are the frames the engine already materialized. `get_mismatches(column)` narrows the changed set to one column with both values side by side.

In [ ]:
print("compared:", result.compared_columns)
print(
    f"added={result.added.height}  removed={result.removed.height}  changed={result.changed.height}"
)
print(result.get_mismatches("balance"))

# Output:
# compared: ('status', 'balance', 'tier')
# added=0  removed=0  changed=0
# shape: (0, 3)
# ┌─────────┬────────────────┬────────────────┐
# │ user_id ┆ balance_source ┆ balance_target │
# │ ---     ┆ ---            ┆ ---            │
# │ i64     ┆ f64            ┆ f64            │
# ╞═════════╪════════════════╪════════════════╡
# └─────────┴────────────────┴────────────────┘

## 4. Global defaults versus per-column overrides

`default_*` on `DiffConfig` apply to every column. A `DiffRule` on a named column wins. The balance tolerance can live on the config; a tighter per-column override still covers the two-cent drift.

In [ ]:
inherited = DiffConfig(
    primary_keys=["user_id"],
    default_absolute_tolerance=0.05,
    default_null_values=["N/A"],
    rules=[
        DiffRule(column_names=["legacy_id"], rename_to="user_id"),
        DiffRule(column_names=["balance"], regex_replace={"\\$": ""}, cast_to="Float64"),
    ],
)
print(
    "inherited default 0.05:",
    DiffEngine(inherited, source.lazy(), target.lazy()).run().summary.is_match,
)

tighter = DiffConfig(
    primary_keys=["user_id"],
    default_absolute_tolerance=0.05,
    default_null_values=["N/A"],
    rules=[
        DiffRule(column_names=["legacy_id"], rename_to="user_id"),
        DiffRule(
            column_names=["balance"],
            regex_replace={"\\$": ""},
            cast_to="Float64",
            absolute_tolerance=0.03,
        ),
    ],
)
print(
    "per-column 0.03:",
    DiffEngine(tighter, source.lazy(), target.lazy()).run().summary.is_match,
)

# Output:
# inherited default 0.05: True
# per-column 0.03: True

## 5. Isolate a real discrepancy

Drop the tolerance and the two-cent drift on row 2 surfaces. `get_mismatches` keeps only that column.

In [ ]:
strict = DiffConfig(
    primary_keys=["user_id"],
    rules=[
        DiffRule(column_names=["legacy_id"], rename_to="user_id"),
        DiffRule(column_names=["balance"], regex_replace={"\\$": ""}, cast_to="Float64"),
        DiffRule(column_names=["status"], null_values=["N/A"], treat_null_as_equal=True),
    ],
)
drifted = DiffEngine(strict, source.lazy(), target.lazy()).run()
print(drifted.summary.report_summary)
print(drifted.get_mismatches("balance"))

# Output:
# Veridelta Execution Summary
# ===========================
# Status:        FAILED
# Match Rate:    66.67%
# Source Rows:   3
# Target Rows:   3
# Volume Shift:  +0 rows
#
# Row-Level Discrepancies:
# ---------------------------
# Added:         0
# Removed:       0
# Changed:       1
# Total Issues:  1
#
# Top Column-Level Drifts:
# ---------------------------
# - balance: 1 mismatches
#
# shape: (1, 3)
# ┌─────────┬────────────────┬────────────────┐
# │ user_id ┆ balance_source ┆ balance_target │
# │ ---     ┆ ---            ┆ ---            │
# │ i64     ┆ f64            ┆ f64            │
# ╞═════════╪════════════════╪════════════════╡
# │ 2       ┆ 20.5           ┆ 20.48          │
# └─────────┴────────────────┴────────────────┘

## 6. Categorical crosswalks

The modern system stores abbreviated tier codes. `value_map` translates source labels before comparison; it is a declared correspondence, not a fuzzy match.

In [ ]:
coded = target.with_columns(tier=pl.Series(["PRM", "STD", "ENT"]))
print(
    "without map, changed:",
    DiffEngine(config, source.lazy(), coded.lazy()).run().summary.changed_count,
)

mapped = DiffConfig(
    primary_keys=["user_id"],
    rules=[
        *config.rules,
        DiffRule(
            column_names=["tier"],
            value_map={"Premium": "PRM", "Standard": "STD", "Enterprise": "ENT"},
        ),
    ],
)
print(
    "with map, changed:",
    DiffEngine(mapped, source.lazy(), coded.lazy()).run().summary.changed_count,
)

# Output:
# without map, changed: 3
# with map, changed: 0

The same comparison as a pipeline is [2. YAML and CLI](02_yaml_and_cli.ipynb). Field-level reference is the [Configuration Guide](../configuration.md).